# 03 — Reproducible Group-Based Splitting

This notebook creates balanced train/validation/test subsets while keeping all questions from an article in exactly one partition. It uses a fixed random seed and saves split metadata so results can be reproduced.

## 1. Load cleaned data

We combine official train and dev question rows into one candidate pool, then create our own article-group train/validation/test partitions. This follows the proposal's requirement for a reproducible split and avoids treating the official dev file as an untouched final test set.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
PROCESSED, SPLITS = ROOT/"data"/"processed", ROOT/"data"/"splits"
SPLITS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED/"questions_clean.csv")
df["answers"] = df["answers"].apply(json.loads)
df["is_answerable"] = df["is_answerable"].astype(bool)
print(df.shape, df.article_title.nunique())
display(df.groupby(["source_split","is_answerable"]).size().unstack(fill_value=0))

## 2. Split by article group

`GroupShuffleSplit` ensures that all questions associated with a given article title go into the same partition. We first reserve a test group, then reserve validation groups from the remaining pool.

The exact row proportions can differ from 70/15/15 because articles contain different numbers of questions. This is preferable to leaking related questions across partitions.

In [ ]:
# First: hold out approximately 15% of article groups for test.
gss_test = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
dev_idx, test_idx = next(gss_test.split(df, groups=df["article_title"]))
development_pool = df.iloc[dev_idx].copy()
test_pool = df.iloc[test_idx].copy()

# Second: hold out validation groups from the remaining development pool.
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=SEED+1)
train_idx, val_idx = next(
    gss_val.split(development_pool, groups=development_pool["article_title"])
)
train_pool = development_pool.iloc[train_idx].copy()
val_pool = development_pool.iloc[val_idx].copy()

for name, part in [("train",train_pool),("validation",val_pool),("test",test_pool)]:
    print(name, "rows:",len(part),"articles:",part.article_title.nunique())

## 3. Verify there is no article overlap

This is a hard validation check. If any article title appears in more than one partition, stop and investigate before proceeding.

In [ ]:
parts = {"train":train_pool, "validation":val_pool, "test":test_pool}
for a, b in [("train","validation"),("train","test"),("validation","test")]:
    overlap = set(parts[a].article_title) & set(parts[b].article_title)
    print(f"{a} ∩ {b}: {len(overlap)} articles")
    assert not overlap, f"Article leakage detected between {a} and {b}" 

## 4. Create balanced subsets after grouping

The proposal targets approximately 6,000–10,000 examples, with a fallback of 2,000–3,000 if needed. We target 8,000 total with a 70/15/15 split and 50/50 answerability balance within each split. If any partition lacks enough examples in one class, the code reduces the target automatically and reports the achieved counts.

This is a controlled research subset, not the entire SQuAD dataset.

In [ ]:
TARGET_TOTAL = 500  # Max 500 examples
SPLIT_FRACTIONS = {"train":0.70, "validation":0.15, "test":0.15}

def balanced_sample(pool, target_total, seed):
    # Equal target per class; reduce to what is available in this article-group pool.
    per_class_target = target_total // 2
    counts = pool["is_answerable"].value_counts()
    n_pos = min(per_class_target, int(counts.get(True, 0)))
    n_neg = min(per_class_target, int(counts.get(False, 0)))
    n_each = min(n_pos, n_neg)
    sampled = pd.concat([
        pool[pool.is_answerable == True].sample(n=n_each, random_state=seed),
        pool[pool.is_answerable == False].sample(n=n_each, random_state=seed+1),
    ], ignore_index=True)
    return sampled.sample(frac=1, random_state=seed).reset_index(drop=True)

sampled = {}
for i, (name, pool) in enumerate(parts.items()):
    target = int(TARGET_TOTAL * SPLIT_FRACTIONS[name])
    sampled[name] = balanced_sample(pool, target, SEED+i)

for name, part in sampled.items():
    print("\n",name, "rows:",len(part))
    print(part.is_answerable.value_counts().rename(index={True:"answerable",False:"unanswerable"}))

## 5. Recheck article separation after sampling

Sampling cannot introduce new articles, but verify the saved subsets still have disjoint article groups.

In [ ]:
for a, b in [("train","validation"),("train","test"),("validation","test")]:
    overlap = set(sampled[a].article_title) & set(sampled[b].article_title)
    print(a, b, "overlap:", len(overlap))
    assert len(overlap) == 0

## 6. Save splits and metadata

We save CSV and JSONL versions. A metadata file records the seed, requested target, actual counts and grouping policy. Keep this file with your experimental results.

In [ ]:
for name, part in sampled.items():
    export = part.copy()
    export["answers"] = export["answers"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    export.to_csv(SPLITS/f"{name}.csv", index=False, encoding="utf-8")
    with (SPLITS/f"{name}.jsonl").open("w", encoding="utf-8") as f:
        for row in part.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False, default=str)+"\n")

metadata = {
    "seed": SEED,
    "requested_total": TARGET_TOTAL,
    "split_fractions": SPLIT_FRACTIONS,
    "group_column": "article_title",
    "sampling": "balanced by is_answerable after article-group split",
    "actual_counts": {name:len(part) for name,part in sampled.items()},
    "class_counts": {
        name: {str(k):int(v) for k,v in part.is_answerable.value_counts().items()}
        for name,part in sampled.items()
    },
}
with (SPLITS/"split_metadata.json").open("w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata, indent=2))

## 7. Important experimental note

For the retrieval experiment, the original SQuAD label describes answerability from the paired source passage. Our BM25 system will retrieve from a split-specific passage corpus. We will measure whether the paired source passage appears in top-k and will state this limitation clearly. The label is a benchmark proxy, not a human annotation of every possible retrieved top-k evidence set.

**Next:** Notebook 04 builds BM25 indexes over unique passages for each split.